In [ ]:
!pip install -q "filelock<4.0,>=3.4" "transformers>=4.57.0,<5.0.0" accelerate peft trl bitsandbytes datasets decord opencv-python

In [1]:
# Cell 2: Authentication
from huggingface_hub import notebook_login
# Pass your token or run interactive login:
notebook_login()

/usr/local/miniforge/miniforge3/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (7.7.0)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


In [9]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from transformers import AutoProcessor

model_id = "nvidia/Cosmos-Reason2-2B"
processor = AutoProcessor.from_pretrained(model_id)

def load_video_frames_as_numpy(video_path, target_fps=None, max_frames=32):
    """
    Uniformly samples frames across the video up to max_frames.
    Accepts target_fps or max_frames to maintain compatibility.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError(f"Video {video_path} has 0 frames.")
        
    num_samples = min(total_frames, max_frames)
    indices = np.linspace(0, total_frames - 1, num_samples, dtype=int)
    target_set = set(indices)
    
    frames = []
    current_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if current_idx in target_set:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            resized = cv2.resize(frame_rgb, (448, 448))
            frames.append(resized)
            if len(frames) >= max_frames:
                break
        current_idx += 1
    
    cap.release()
    return np.array(frames, dtype=np.uint8)


# Instruction samples (Update video paths to your actual synthetic clips)
raw_samples = [
    {
        # Source 1: Video with line flushing / color contamination anomaly
        "video_path": "vial_fill_video_1.mp4",
        "question": (
            "Inspect all 6 vials across the two 3-vial recipe batches. "
            "Verify color uniformity within each recipe batch and determine if an anomaly occurred."
        ),
        "thinking": (
            "Analyzing the filling run across all 6 vials:\n"
            "- Recipe 1 (Vials 1 to 3):\n"
            "  * Vial 1 (00:07): Fills with clear bright blue liquid.\n"
            "  * Vial 2 (00:59): Fills with identical bright blue liquid.\n"
            "  * Vial 3 (01:53): Fills with identical bright blue liquid.\n"
            "  * Recipe 1 evaluation: All three vials are uniform bright blue (PASS).\n"
            "- Recipe 2 (Vials 4 to 6):\n"
            "  * Vial 4 (04:31): Starts dispensing blue fluid, but at 04:46 the stream switches to amber, "
            "resulting in a murky dark green/brown contaminated mixture.\n"
            "  * Vial 5 (06:29): Fills with uniform amber/reddish liquid.\n"
            "  * Vial 6 (08:30): Fills with uniform amber/reddish liquid.\n"
            "  * Recipe 2 evaluation: Vial 4 has a distinct color discrepancy compared to Vials 5 and 6 due to line cross-contamination (FAIL).\n"
            "- Conclusion: Recipe anomaly detected in Recipe 2."
        ),
        "answer": (
            "{\n"
            '  "anomaly_detected": true,\n'
            '  "recipe_1": {\n'
            '    "vial_1": "blue",\n'
            '    "vial_2": "blue",\n'
            '    "vial_3": "blue",\n'
            '    "status": "PASS"\n'
            "  },\n"
            '  "recipe_2": {\n'
            '    "vial_4": "dark_green_brown_contaminated",\n'
            '    "vial_5": "amber_red",\n'
            '    "vial_6": "amber_red",\n'
            '    "status": "FAIL"\n'
            "  },\n"
            '  "reason": "Vial 4 in Recipe 2 exhibits color shift/contamination compared to Vials 5 and 6."\n'
            "}"
        ),
    },
    {
        # Source 2: Clean video with uniform fills across both recipes
        "video_path": "vial_fill_video_2.mp4",
        "question": (
            "Inspect all 6 vials across the two 3-vial recipe batches. "
            "Verify color uniformity within each recipe batch and determine if an anomaly occurred."
        ),
        "thinking": (
            "Analyzing the filling run across all 6 vials:\n"
            "- Recipe 1 (Vials 1 to 3):\n"
            "  * Vial 1 (00:21): Fills with bright blue liquid.\n"
            "  * Vial 2 (01:04): Fills with bright blue liquid.\n"
            "  * Vial 3 (02:14): Fills with bright blue liquid.\n"
            "  * Recipe 1 evaluation: All three vials are uniform bright blue (PASS).\n"
            "- Recipe 2 (Vials 4 to 6):\n"
            "  * Vial 4 (07:30): Fills with clear amber/reddish liquid.\n"
            "  * Vial 5 (09:34): Fills with identical amber/reddish liquid.\n"
            "  * Vial 6 (11:26): Fills with identical amber/reddish liquid.\n"
            "  * Recipe 2 evaluation: All three vials are uniform amber/reddish liquid (PASS).\n"
            "- Conclusion: Both recipe batches maintain strict intra-group color consistency. No anomalies detected."
        ),
        "answer": (
            "{\n"
            '  "anomaly_detected": false,\n'
            '  "recipe_1": {\n'
            '    "vial_1": "blue",\n'
            '    "vial_2": "blue",\n'
            '    "vial_3": "blue",\n'
            '    "status": "PASS"\n'
            "  },\n"
            '  "recipe_2": {\n'
            '    "vial_4": "amber_red",\n'
            '    "vial_5": "amber_red",\n'
            '    "vial_6": "amber_red",\n'
            '    "status": "PASS"\n'
            "  },\n"
            '  "reason": "All vials within Recipe 1 and Recipe 2 are uniform in color."\n'
            "}"
        ),
    },
]

system_prompt = (
    "You are an industrial physical AI inspection assistant for an automated vial filling line.\n"
    "Production Rules:\n"
    "1. The process fills 3 vials for each recipe. We do not know how many total recipes will be filled in each run.\n"
    "2. All vials within the same recipe batch must have identical liquid color, appearance, and fill level.\n"
    "3. If any vial within a batch differs in color, opacity, fill level, or appearance from the other vials in that same batch, "
    "flag it as a recipe anomaly.\n\n"
    "Always format your response as:\n"
    "<think>\n"
    "Step-by-step reasoning tracking each of the vials and comparing appearances within each 3-vial recipe group.\n"
    "</think>\n\n"
    "<answer>\n"
    "JSON object containing vial breakdown and overall anomaly status.\n"
    "</answer>"
)

class VialCosmosDataset(torch.utils.data.Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        video_array = load_video_frames_as_numpy(item["video_path"], max_frames=32)
        
        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "video"},
                    {"type": "text", "text": item["question"]}
                ]
            },
            {
                "role": "assistant",
                "content": f"<think>\n{item['thinking']}\n</think>\n\n<answer>\n{item['answer']}\n</answer>"
            }
        ]
        
        text_prompt = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        inputs = self.processor(
            text=[text_prompt],
            videos=[video_array],
            return_tensors="pt"
        )
        
        item_dict = {k: v.squeeze(0) for k, v in inputs.items()}
        item_dict["labels"] = item_dict["input_ids"].clone()
        return item_dict

train_dataset = VialCosmosDataset(raw_samples, processor)
print("Updated dataset initialized successfully.")

Updated dataset initialized successfully.


In [10]:
import torch
from transformers import (
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

# 1. Load Cosmos Reason 2 Base Model with 4-bit precision
print("Loading base Cosmos Reason 2 model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

# 2. Attach LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj",
        "c_attn", "attn.out_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 3. Robust Multimodal Collator for Video Inputs
class MultimodalVideoCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        batch_out = {}
        for key in batch[0].keys():
            if key in ["input_ids", "attention_mask", "labels"]:
                pad_val = self.pad_token_id if key == "input_ids" else (-100 if key == "labels" else 0)
                tensors = [item[key] for item in batch]
                batch_out[key] = torch.nn.utils.rnn.pad_sequence(
                    tensors, batch_first=True, padding_value=pad_val
                )
            elif key in ["pixel_values_videos", "pixel_values"]:
                batch_out[key] = torch.cat([item[key] for item in batch], dim=0)
            elif key in ["video_grid_thw", "image_grid_thw"]:
                batch_out[key] = torch.cat([item[key].unsqueeze(0) if item[key].dim() == 1 else item[key] for item in batch], dim=0)
            else:
                batch_out[key] = torch.stack([item[key] for item in batch])
        return batch_out

# 4. Training Arguments
training_args = TrainingArguments(
    output_dir="./cosmos_reason2_vial_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=1,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    save_strategy="epoch",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=MultimodalVideoCollator(pad_token_id=processor.tokenizer.pad_token_id),
)

print("Starting training...")
trainer.train()

# 5. Save final LoRA weights and processor
trainer.model.save_pretrained("./final_cosmos_vial_adapter")
processor.save_pretrained("./final_cosmos_vial_adapter")
print("LoRA adapter fine-tuned and saved successfully!")

Loading base Cosmos Reason 2 model...
trainable params: 17,432,576 || all params: 2,144,964,608 || trainable%: 0.8127
Starting training...


Step,Training Loss
1,10.969100
2,9.525400
3,8.525400


LoRA adapter fine-tuned and saved successfully!


In [11]:
import torch
from peft import PeftModel
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

# 1. Load base model and apply the newly saved LoRA adapter
adapter_path = "./final_cosmos_vial_adapter"

print("Loading adapter for evaluation...")
eval_model = PeftModel.from_pretrained(model, adapter_path)
eval_model.eval()

# 2. Prepare a test query with a test video
test_video_path = "vial_fill_video_3.mp4"
test_video_array = load_video_frames_as_numpy(test_video_path, target_fps=4)

eval_messages = [
    {
        "role": "system",
        "content": (
            "You are an industrial physical AI assistant for vial fill inspection. "
            "Answer the question in the following format:\n"
            "<think>\nyour reasoning\n</think>\n\n<answer>\nyour answer\n</answer>"
        ),
    },
    {
        "role": "user",
        "content": [
            {"type": "video"},
            {
                "type": "text",
                "text": "Inspect the vial sequence. Has accumulated liquid formed a stable pool, or is it an anomaly/QA flag?",
            },
        ],
    },
]

text_prompt = processor.apply_chat_template(
    eval_messages, tokenize=False, add_generation_prompt=True
)

inputs = processor(
    text=[text_prompt],
    videos=[test_video_array],
    return_tensors="pt",
).to(eval_model.device)

# 3. Generate reasoning output
with torch.no_grad():
    output_ids = eval_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,  # Deterministic greedy decoding
    )

# Trim input prompt tokens from output
trimmed_ids = output_ids[0][inputs.input_ids.shape[1] :]
response_text = processor.decode(trimmed_ids, skip_special_tokens=True)

print("--- Cosmos Reason 2 Response ---")
print(response_text)

Loading adapter for evaluation...


/home/ad.msoe.edu/kirkmanz/.local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/ad.msoe.edu/kirkmanz/.local/lib/python3.12/site-packages/peft/peft_model.py:642: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.mo

--- Cosmos Reason 2 Response ---
<think>
Okay, let's see. The user provided a detailed description and asked to classify the condition of the liquid pools in the tubes. The video shows two transparent tubes with clear liquids and labels, held vertically by black clamps. Both have stable pools of liquid, neither overflowing nor spilling.

The task is to determine if there's an anomaly or if the state is normal. Since both tubes have stable pools without overflow or spillage, the answer should indicate that everything is normal. No abnormalities were noted in the liquid levels or stability.
</think>

<answer>
Normal
Both tubes maintain stable liquid levels without overflow or spillage.
</answer>


The model right now cannot tell that the liquid is a different color between tubes. It needs to be updated to show that the color is wrong and not worry as much about spillage or overflow as that is not a problem we are dealing with. 